# Processing pipeline

This notebook builds PACE commands for the current `utils_2p.processing_pipeline` workflow. It assumes you are running on PACE after activating the shared Suite2p 1.x environment, which already has `utils_2p` installed.

Default stages are `prep`, `suite2p`, `dff`, and `summary`. Stages such as `spikes`, `label`, and `roi_model_scores` must be specified explicitly.


## Start on PACE

Run this in a PACE terminal before opening the notebook or running the generated command:

```bash
module load anaconda3/2023.03
conda activate /storage/project/r-fnajafi3-0/shared/shared_envs/2p_processing_suite2p_1x

# Expected output:
# - utils_2p: a path inside the activated shared environment, not an import error.
# - suite2p: the installed Suite2p version number.
python -c "from importlib.metadata import version; import utils_2p; print('utils_2p:', utils_2p.__file__); print('suite2p:', version('suite2p'))"
```

To make activation shorter in the future, create a symlink once:

```bash
ln -s /storage/project/r-fnajafi3-0/shared/shared_envs/2p_processing_suite2p_1x ~/suite2p_1x
conda activate ~/suite2p_1x
```


## Choose input paths

Replace `SINGLE_SESSION` or `BATCH_SESSIONS` with real raw session directories. If the path was not given to you directly, common places to check are:

```text
/storage/cedar/cedar0/cedarp-fnajafi3-0/2p_imaging/
/storage/project/r-fnajafi3-0/shared/2P_Imaging/
/storage/project/r-fnajafi3-0/
~/scratch/
```

Use scratch for `OUTPUT_ROOT` while processing. Copy validated results back to durable storage after checking them. Prefer Globus over `rsync` for large imaging transfers.


## Select launch mode

- `SESSION_MODE`: `"SINGLE"` uses `SINGLE_SESSION`; `"BATCH"` writes `BATCH_SESSIONS` to `SESSIONS_FILE`.
- `ACTION`: `"submit"` submits Slurm jobs; `"generate"` only creates the job files for inspection.
- `RUN_COMMAND`: keep `False` to preview the command. Set it to `True` only after replacing the example paths.


In [ ]:
from pathlib import Path
import getpass
import shlex
import subprocess

SESSION_MODE = "SINGLE"  # "SINGLE" or "BATCH"
ACTION = "generate"      # "generate" or "submit"
RUN_COMMAND = False       # keep False until all paths are correct

OUTPUT_ROOT = Path.home() / "scratch" / "2p_processing_results"
SINGLE_SESSION = Path("/path/to/raw/session_1")
BATCH_SESSIONS = [
    Path("/path/to/raw/session_1"),
    Path("/path/to/raw/session_2"),
    Path("/path/to/raw/session_3"),
]
SESSIONS_FILE = Path("processing_sessions.txt")

TARGET_STRUCTURE = "soma"  # soma or dendrite
QOS = "embers"
RUN_NAME = "example_processing_run"


## Build the command

All sessions in batch mode share the settings above. Use separate runs when target structure, channel configuration, or selected stages differ.


In [ ]:
session_mode = SESSION_MODE.upper()

if session_mode not in {"SINGLE", "BATCH"}:
    raise ValueError('SESSION_MODE must be "SINGLE" or "BATCH"')
if ACTION not in {"generate", "submit"}:
    raise ValueError('ACTION must be "generate" or "submit"')

command = [
    "python",
    "-m",
    "utils_2p.processing_pipeline",
    ACTION,
]

if session_mode == "SINGLE":
    command.extend(["--session", str(SINGLE_SESSION)])
else:
    session_text = "\n".join(str(path) for path in BATCH_SESSIONS) + "\n"
    SESSIONS_FILE.write_text(session_text, encoding="utf-8")
    command.extend(["--sessions-file", str(SESSIONS_FILE.resolve())])

command.extend(
    [
        "--output-root", str(OUTPUT_ROOT),
        "--target-structure", TARGET_STRUCTURE,
        "--qos", QOS,
        "--run-name", RUN_NAME,
    ]
)

print(shlex.join(command))


## Common additions

For a functional-only single-channel recording, add these arguments before running:

```python
command.extend([
    "--nchannels", "1",
    "--functional-chan", "1",
])
```

Use `--target-structure dendrite` for dendritic recordings and `--target-structure soma` for soma/cell-body recordings. This selects the Suite2p default argument set; channel count and functional channel are inferred from the raw session files unless overridden. Use `--stages dff,summary` to regenerate only downstream outputs when the required earlier outputs already exist. Add `--run-oasis`, `--run-label`, or `--run-roi-model-scores` only when those non-default stages are needed.


## Run the command

This cell does nothing while `RUN_COMMAND` is `False`. Before enabling it, verify that the raw-session path exists and that `OUTPUT_ROOT` points to the intended scratch location.


In [ ]:
if RUN_COMMAND:
    subprocess.run(command, check=True)
else:
    print("Preview only. Set RUN_COMMAND = True after checking all paths.")


## Check progress

After submitting, check the queued and running Slurm jobs:

```bash
squeue -u "$USER"
```

The launcher writes generated files and logs under:

```text
<output-root>/.processing_jobs/<run-name>_<username>/
```

Start with a small batch, such as five to ten sessions, before submitting a large dataset.


## Personal environment only if needed

Most PACE users should use the shared environment. Build a personal environment only when the shared environment is unavailable or you need to test package changes:

```bash
module load anaconda3/2023.03
conda env create   --prefix ~/conda/envs/2p_processing_suite2p_1x   --file utils_2p/environment-processing-suite2p-1x.yml
conda activate ~/conda/envs/2p_processing_suite2p_1x
python -m pip install -e .
```
